# Aprendizado de Máquina — Aula prática 02

## Regressão Linear e Regularização

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Na Aula prática 01 controlamos a flexibilidade de um modelo pelo **grau de um
polinômio**: subíamos o grau, a variância crescia, o viés caía.

Esta aula troca a maneira como enxergamos a complexidade. 
Mantemos a família de modelos fixa — regressão linear com
todas as covariáveis — e controlamos a complexidade **encolhendo os coeficientes**
na direção do zero.

O roteiro:

1. montamos o MQO (Mínimos Quadrados Ordinários) e vemos, com dados reais, as **três limitações** que ele sofre em
   dimensão alta (Seção 3);
2. introduzimos **Ridge** e **Lasso** como duas respostas ao mesmo problema
   (Seções 4 e 5);
3. entendemos, com duas figuras, **por que só o Lasso zera coeficientes**
   (Seção 6) — a pergunta central da aula;
4. verificamos, em dados simulados, se o Lasso realmente **descobre** quais
   coeficientes são nulos (Seção 7);
5. e comparamos os três nos supercondutores (Seção 8), num conjunto de teste
   que nenhum deles viu.

Ao longo do caminho, um aviso de notação: o que as notas chamam de $\lambda$, o
`scikit-learn` chama de `alpha`. É a mesma coisa.

### Objetivos

Ao final deste notebook você deve ser capaz de:

- calcular $\hat\beta^{MQO}=(X^\top X)^{-1}X^\top Y$ e reconhecer quando essa
  expressão **não faz sentido**;
- diagnosticar colinearidade e o caso $p>n$ em dados reais;
- ajustar Ridge e Lasso e ler um **caminho de coeficientes**;
- explicar, com a geometria, por que o Lasso produz esparsidade e o Ridge não;
- comparar MQO, Ridge e Lasso fora da amostra, e reconhecer quando a
  regularização **não** ajuda.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

Os novos objetos desta aula. Note `skl.ElasticNet`: no `scikit-learn`, Ridge, Lasso
e Elastic Net são **a mesma função** com valores diferentes de `l1_ratio`. Vamos
explorar isso.

In [ ]:
import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor as VIF

import warnings
warnings.filterwarnings("ignore")

---
## 2. Mínimos quadrados

Voltamos aos supercondutores da Aula prática 01: 21 263 materiais, 81 medidas
físico-químicas por material, e a temperatura crítica como resposta.

O arquivo `superconductivity.csv` **não é baixado da internet**. O notebook o
procura em dois lugares, nesta ordem: a pasta onde este `.ipynb` está e, se você
tiver o repositório do curso, `recursos/dados/`. Basta que ele esteja num dos
dois — se estiver em outro lugar, ajuste `_lugares` na célula abaixo.

In [ ]:
import os

_nome = "superconductivity.csv"

# procura em dois lugares, sem baixar nada da internet: a pasta deste
# notebook primeiro ou então ../../recursos/dados/
_lugares = [_nome, os.path.join("..", "..", "recursos", "dados", _nome)]
_caminho = next((c for c in _lugares if os.path.exists(c)), None)

if _caminho is None:
    raise FileNotFoundError(
        f"nao encontrei '{_nome}'. Procurei nesta pasta e em "
        "../../recursos/dados/. Ponha o .csv ao lado deste notebook, "
        "ou mude o caminho se for necessário."
    )

df = pd.read_csv(_caminho)
X_todos = df.drop(columns="critical_temp")
y = df["critical_temp"].values

print("dimensoes de X:", X_todos.shape)
print("dimensoes de y:", y.shape)
print("resposta: critical_temp  (media %.1f K, desvio %.1f K)" % (y.mean(), y.std()))
X_todos.iloc[:, :6].head()

### $\hat\beta$ na mão

As notas deduzem o estimador de mínimos quadrados em forma matricial:

$$\hat\beta^{MQO} = (X^\top X)^{-1} X^\top Y .$$

Vamos calculá-lo literalmente, com cinco covariáveis fisicamente distintas, e
conferir contra o `scikit-learn`. Uma observação numérica: na prática **nunca** se
inverte a matriz — resolve-se o sistema linear $(X^\top X)\beta = X^\top Y$, que é
mais estável e mais rápido. É o que `np.linalg.solve` faz.

In [ ]:
poucas = ["number_of_elements", "mean_atomic_mass", "mean_Density",
          "mean_ThermalConductivity", "mean_Valence"]

X5 = np.column_stack([np.ones(len(df)), X_todos[poucas].values])   # com intercepto

beta_mao = np.linalg.solve(X5.T @ X5, X5.T @ y)

beta_skl = skl.LinearRegression().fit(X_todos[poucas].values, y)

print("na mao (intercepto + 5) :", beta_mao.round(4))
print("scikit-learn            :", np.r_[beta_skl.intercept_, beta_skl.coef_].round(4))
print("diferenca maxima        :", np.abs(beta_mao - np.r_[beta_skl.intercept_, beta_skl.coef_]).max())

Idênticos. O `scikit-learn` não faz mágica nenhuma aqui — é a mesma conta.

E o `statsmodels`, como vimos na aula passada, responde à outra pergunta:

In [ ]:
ajuste_ols = sm.OLS(y, sm.add_constant(X_todos[poucas])).fit()
print(ajuste_ols.summary().tables[1])
print("R^2 =", round(ajuste_ols.rsquared, 4))

As dez versões da massa atômica que a próxima seção vai listar não são medidas
independentes. Vale antecipar o efeito disso com o caso mínimo: acrescentar ao
conjunto `poucas` a covariável `wtd_mean_atomic_mass` — a mesma massa atômica
média, só que ponderada pela composição do material — e ver o que acontece com o
coeficiente da sua quase-gêmea.

In [ ]:
poucas_mais = poucas + ["wtd_mean_atomic_mass"]
ajuste_mais = sm.OLS(y, sm.add_constant(X_todos[poucas_mais])).fit()

lado_a_lado = pd.DataFrame({
    "sem wtd": ajuste_ols.params,
    "com wtd": ajuste_mais.params,
}).round(3)
print(lado_a_lado.to_string())
print()

b_sem = ajuste_ols.params["mean_atomic_mass"]
b_com = ajuste_mais.params["mean_atomic_mass"]
r = X_todos[["mean_atomic_mass", "wtd_mean_atomic_mass"]].corr().iloc[0, 1]

print(f"mean_atomic_mass: {b_sem:+.3f}  ->  {b_com:+.3f}   (mudou de sinal: {b_sem * b_com < 0})")
print(f"erro-padrao dele: {ajuste_ols.bse['mean_atomic_mass']:.3f}  ->  "
      f"{ajuste_mais.bse['mean_atomic_mass']:.3f}")
print(f"correlacao entre as duas massas atomicas: {r:.4f}")
print(f"R^2: {ajuste_ols.rsquared:.4f}  ->  {ajuste_mais.rsquared:.4f}")

**O coeficiente não muda de sinal, mas cresce 40%:** de $+0{,}227$ para $+0{,}319$.
E a covariável nova entra com sinal **oposto**, $-0{,}123$.

As duas medem quase a mesma coisa — correlação $0{,}816$ — e o ajuste passa a
repartir entre elas um efeito que antes era de uma só. Uma sobe, a outra desce, e a
soma continua explicando os mesmos dados: o $R^2$ vai de $0{,}4900$ para $0{,}4937$,
quatro décimos de ponto percentual. Quase nenhuma informação nova entrou, e mesmo
assim os coeficientes se mexeram bastante.

O erro-padrão de `mean_atomic_mass` cresce junto, de $0{,}010$ para $0{,}012$. É a
primeira aparição da Limitação 1 da próxima seção: colinearidade não estraga a predição,
estraga a **leitura** dos coeficientes. Com 81 covariáveis e dez versões de massa
atômica, o que aqui é um deslocamento de 40% vira algo bem pior.

Repare no que **não** aconteceu: nenhum sinal virou. Colinearidade forte o
suficiente inverte sinais, e isso é comum na prática — só que uma correlação de
$0{,}816$ ainda não basta. O que interessa guardar é o mecanismo, não o desfecho
deste caso.

---
## 3. Onde o MQO quebra

As notas listam três limitações do MQO quando $p$ é grande em relação a $n$. Vamos
produzir evidência numérica para cada um.

### Limitação 1 — variância alta por colinearidade

A variância do estimador é $\operatorname{Var}(\hat\beta)=\sigma^2(X^\top X)^{-1}$.
Quando as colunas de $X$ são quase linearmente dependentes, $X^\top X$ fica quase
singular e a inversa "explode".

Estes dados são um caso de laboratório: as 81 covariáveis são
**estatísticas-resumo das mesmas propriedades**. Só de massa atômica há dez
versões — média, média ponderada, média geométrica, entropia, amplitude, desvio...

In [ ]:
print([c for c in X_todos.columns if "atomic_mass" in c])

In [ ]:
XtX = X_todos.values.T @ X_todos.values

print(f"numero de condicionamento de X^T X : {np.linalg.cond(XtX):.3e}")
print(f"(para comparacao, uma matriz bem-condicionada tem condicionamento ~ 1)")

$10^{13}$. Isso significa, grosso modo, que perdemos **13 dígitos** de precisão ao
resolver o sistema — de cerca de 16 que a aritmética de ponto flutuante oferece.
Estamos operando na beira do abismo numérico.

O diagnóstico clássico por covariável é o **VIF** (*variance inflation factor*):
quanto a variância do coeficiente $j$ é inflada pela colinearidade com as demais.
Valores acima de 10 já são considerados problemáticos.

In [ ]:
grupo = [c for c in X_todos.columns if "atomic_mass" in c]
Xg = sm.add_constant(X_todos[grupo]).values

vifs = pd.Series([VIF(Xg, i) for i in range(1, Xg.shape[1])], index=grupo)
vifs.sort_values(ascending=False).round(1)

Ordens de grandeza acima de 10. Na prática: os coeficientes individuais são
instáveis e **não interpretáveis** — trocar uma covariável por outra do mesmo grupo
pode inverter sinais sem alterar as predições. Foi o que o exercício da Seção 2
pediu para você verificar.

### Limitação 2 — não unicidade quando $p > n$

Se $p>n$, a matriz $X^\top X$ é **singular** e a fórmula do MQO simplesmente não
tem solução única: existem infinitos $\beta$ com RSS igual a zero.

Nossos dados têm $n \gg p$, então vamos forçar a situação: 50 materiais, 81
covariáveis.

In [ ]:
rng = np.random.default_rng(0)
idx = rng.choice(len(df), size=50, replace=False)

Xp = X_todos.values[idx]
yp = y[idx]

print(f"n = {Xp.shape[0]}, p = {Xp.shape[1]}   ->  p > n")
print(f"posto de X^T X : {np.linalg.matrix_rank(Xp.T @ Xp)}  (precisaria ser {Xp.shape[1]})")

Posto 50 numa matriz $81\times81$: singular, como previsto. O `np.linalg.lstsq`
ainda devolve *uma* resposta — a de **norma mínima** — mas é uma escolha
arbitrária entre infinitas. Vamos exibir outra, somando um vetor do núcleo de $X$:

In [ ]:
beta_min = np.linalg.lstsq(Xp, yp, rcond=None)[0]

_, s, vt = np.linalg.svd(Xp)
nucleo = vt[np.sum(s > 1e-8):]          # base do nucleo de X
beta_alt = beta_min + 100 * nucleo[0]   # outra solucao, igualmente valida

print(f"dimensao do nucleo de X : {nucleo.shape[0]}")
print()
print(f"RSS de beta_min : {np.sum((yp - Xp @ beta_min)**2):.3e}     norma: {np.linalg.norm(beta_min):7.2f}")
print(f"RSS de beta_alt : {np.sum((yp - Xp @ beta_alt)**2):.3e}     norma: {np.linalg.norm(beta_alt):7.2f}")

Os dois vetores ajustam os 50 pontos **perfeitamente** (RSS da ordem de $10^{-19}$,
ou seja, zero), e são completamente diferentes um do outro. "O" estimador de
mínimos quadrados não existe aqui — existe um subespaço de dimensão 31 deles.

E o ajuste perfeito no treino não vale nada:

In [ ]:
fora = np.setdiff1d(np.arange(len(df)), idx)[:5000]

print(f"EQM no treino (50 pontos) : {mean_squared_error(yp, Xp @ beta_min):.3e}")
print(f"EQM fora da amostra       : {mean_squared_error(y[fora], X_todos.values[fora] @ beta_min):.3e}")
print(f"EQM do preditor constante : {y[fora].var():.3e}")

Erro de treino **zero**, erro fora da amostra pior que o de um modelo que sempre
chuta a média. É o superajuste da Aula 01 na sua forma mais pura — e é exatamente
o que o [ISLP] alerta no Capítulo 6: nunca use $R^2$ ou RSS de treino como evidência
de ajuste quando $p$ é grande. Aqui o $R^2$ de treino é 1, e é 1 por vacuidade.

### Limitação 3 — interpretabilidade

Ainda que tudo desse certo numericamente, 81 coeficientes não são uma explicação.
O que gostaríamos é de um modelo que dissesse: *"destas 81 medidas, estas 7
importam"*. É precisamente isso que o Lasso vai entregar.

### A resposta: reduzir a complexidade efetiva

Duas famílias de solução, segundo as notas:

- **selecionar um subconjunto** de covariáveis (melhor subconjunto, *forward* e
  *backward stepwise*). O melhor subconjunto exige ajustar $2^{81}$ modelos — mais
  do que átomos no universo observável. Inviável;
- **encolher os coeficientes** (regularização). É o caminho desta aula.

E há uma vantagem escondida na segunda: o Lasso faz as duas coisas ao mesmo tempo.

---
## 4. Regressão Ridge ($\ell_2$)

A ideia da regularização é somar ao RSS uma penalização que cresce com o tamanho
dos coeficientes. No Ridge, a penalização é a norma $\ell_2$:

$$\hat\beta^{Ridge}_\lambda \;=\; \arg\min_\beta \ \underbrace{\sum_{i=1}^n\big(y_i-\beta^\top x_i\big)^2}_{\text{RSS: quero ajustar}} \;+\; \underbrace{\lambda\sum_{j=1}^p \beta_j^2}_{\text{quero coeficientes pequenos}}.$$

$\lambda \ge 0$ arbitra o conflito: $\lambda=0$ devolve o MQO; $\lambda\to\infty$
empurra todos os coeficientes para zero.

### Por que padronizar é obrigatório

O termo $\sum_j \beta_j^2$ soma coeficientes de covariáveis medidas em unidades
diferentes. Multiplicar uma covariável por 1000 divide seu coeficiente por 1000 e
**muda a penalização** — o modelo passaria a depender das unidades em que os dados
foram registrados, o que é indefensável. Por isso padronizamos as colunas antes.

O `skl.ElasticNet` não padroniza sozinho. Seguindo o [ISLP], fazemos à mão nesta
primeira passagem — e depois mostramos a forma correta, com `Pipeline`.

In [ ]:
X = X_todos.values

Xs = (X - X.mean(axis=0)) / X.std(axis=0)      # colunas com media 0 e desvio 1

lambdas = 10 ** np.linspace(4, -2, 100) / y.std()

print("grade de lambdas: de %.3e a %.3e" % (lambdas.max(), lambdas.min()))
print("Xs: media ~ %.1e, desvio ~ %.4f" % (Xs.mean(), Xs.std()))

`skl.ElasticNet.path` ajusta o modelo para **toda** a grade de $\lambda$ de uma vez,
devolvendo uma matriz $p \times 100$: uma coluna de coeficientes por valor de
$\lambda$. Ridge corresponde a `l1_ratio=0`.

In [ ]:
caminho_ridge = skl.ElasticNet.path(Xs, y, l1_ratio=0.0, alphas=lambdas)[1]

print("formato do caminho:", caminho_ridge.shape, " (covariaveis x lambdas)")

In [ ]:
tabela_ridge = pd.DataFrame(caminho_ridge.T, columns=X_todos.columns,
                            index=-np.log(lambdas))
tabela_ridge.index.name = "-log(lambda)"
tabela_ridge.iloc[::20, :5].round(4)

Vamos ver o caminho inteiro. São 81 curvas: desenhamos todas em cinza para dar a
forma geral, e destacamos cinco covariáveis para poder acompanhar trajetórias
individuais.

In [ ]:
destaque = ["number_of_elements", "wtd_std_atomic_mass", "wtd_std_fie",
            "wtd_std_Density", "wtd_std_FusionHeat"]

fig, ax = subplots(figsize=(9, 6))
ax.plot(-np.log(lambdas), caminho_ridge.T, c="lightgray", lw=0.8)
for nome in destaque:
    j = list(X_todos.columns).index(nome)
    ax.plot(-np.log(lambdas), caminho_ridge[j], lw=2, label=nome)
ax.axhline(0, c="k", lw=0.8)
ax.set_xlabel(r"$-\log(\lambda)$   ($\lambda$ diminui $\rightarrow$)")
ax.set_ylabel("coeficientes padronizados")
ax.set_title("Caminho de coeficientes — Ridge")
ax.legend(fontsize=8, loc="upper left");

Leia a figura da **esquerda para a direita**: à esquerda $\lambda$ é enorme e todos
os coeficientes estão esmagados contra o zero; à direita $\lambda\to 0$ e
recuperamos o MQO, com coeficientes grandes e de sinais variados.

Duas coisas a notar:

1. o afastamento do zero é **gradual e simultâneo** — todas as 81 curvas saem do
   zero juntas, e nenhuma volta a tocá-lo;
2. na ponta direita as curvas se espalham violentamente (alguns coeficientes
   passam de $\pm 30$). Essa é a variância alta da Seção 3, agora visível: o Ridge
   está domando exatamente esse comportamento.

Vamos quantificar o encolhimento pela norma $\ell_2$ dos coeficientes em dois
valores de $\lambda$:

In [ ]:
for k in [20, 60, 95]:
    beta_k = caminho_ridge[:, k]
    print(f"lambda = {lambdas[k]:10.4f}   ||beta||_2 = {np.linalg.norm(beta_k):8.2f}"
          f"   coeficientes nao-nulos: {(np.abs(beta_k) > 1e-8).sum()} de 81")

O encolhimento é claro — mas repare na última coluna: **nenhum coeficiente é
exatamente zero**, em nenhum valor de $\lambda$. O Ridge encolhe, mas não seleciona.
Guarde isso.

### A forma correta: `Pipeline`

Padronizar à mão funcionou, mas é perigoso: na hora de fazer validação cruzada,
padronizar antes de separar as dobras vaza informação do conjunto de validação para
o de treino. O `Pipeline` resolve isso encapsulando a padronização como parte do
modelo — ela é reajustada dentro de cada dobra. É o assunto da Aula 07, mas a
prática começa agora.

In [ ]:
ridge_pipe = Pipeline([
    ("escala", StandardScaler()),
    ("ridge",  skl.Ridge(alpha=lambdas[60] * len(y))),   # ver nota abaixo
])
ridge_pipe.fit(X, y)

print("||beta||_2 pelo Pipeline :", round(np.linalg.norm(ridge_pipe.named_steps["ridge"].coef_), 2))
print("||beta||_2 a mao         :", round(np.linalg.norm(caminho_ridge[:, 60]), 2))

> **Uma pegadinha de escala.** `skl.Ridge` minimiza
> $\|y-X\beta\|^2 + \alpha\|\beta\|^2$, enquanto `skl.ElasticNet` minimiza
> $\frac{1}{2n}\|y-X\beta\|^2 + \alpha\|\beta\|^2$ — o RSS dividido por $2n$. Por
> isso o fator `len(y)` acima. É o tipo de detalhe que não muda a teoria e arruína
> uma comparação numérica; ao trocar de função, confira a documentação da
> convenção adotada.

---
## 5. O Lasso ($\ell_1$)

Mesma estrutura, uma troca no expoente da penalização:

$$\hat\beta^{Lasso}_\lambda \;=\; \arg\min_\beta \ \sum_{i=1}^n\big(y_i-\beta^\top x_i\big)^2 \;+\; \lambda\sum_{j=1}^p \big|\beta_j\big| .$$

Trocamos $\beta_j^2$ por $|\beta_j|$. Parece uma diferença cosmética. Não é.

In [ ]:
lambdas_lasso, caminho_lasso = skl.Lasso.path(Xs, y, n_alphas=100)[:2]

print("formato do caminho:", caminho_lasso.shape)

In [ ]:
fig, ax = subplots(figsize=(9, 6))
ax.plot(-np.log(lambdas_lasso), caminho_lasso.T, c="lightgray", lw=0.8)
for nome in destaque:
    j = list(X_todos.columns).index(nome)
    ax.plot(-np.log(lambdas_lasso), caminho_lasso[j], lw=2, label=nome)
ax.axhline(0, c="k", lw=0.8)
ax.set_xlabel(r"$-\log(\lambda)$   ($\lambda$ diminui $\rightarrow$)")
ax.set_ylabel("coeficientes padronizados")
ax.set_title("Caminho de coeficientes — Lasso")
ax.legend(fontsize=8, loc="upper left");

Compare com a figura do Ridge. Aqui as curvas **descolam do zero uma de cada vez**,
em pontos diferentes do eixo. Cada "descolamento" é uma covariável entrando no
modelo. À esquerda, o modelo tem zero covariáveis; conforme $\lambda$ diminui,
elas entram na ordem de importância.

Essa contagem é o gráfico mais eloquente da aula:

In [ ]:
nao_nulos = (np.abs(caminho_lasso) > 1e-10).sum(axis=0)

fig, ax = subplots(figsize=(9, 5))
ax.plot(-np.log(lambdas_lasso), nao_nulos, "o-", ms=3)
ax.set_xlabel(r"$-\log(\lambda)$   ($\lambda$ diminui $\rightarrow$)")
ax.set_ylabel("coeficientes não-nulos")
ax.set_title("O Lasso seleciona covariáveis")
ax.grid(alpha=.3)

print(f"minimo de nao-nulos: {nao_nulos.min()}   maximo: {nao_nulos.max()}  (de 81)")

De 0 a 63 covariáveis, conforme $\lambda$. O mesmo gráfico para o Ridge seria uma
linha reta constante em 81.

**O Lasso ajusta e seleciona ao mesmo tempo.** Ele resolve, de quebra, a Limitação 3 da
Seção 3 — e faz isso sem precisar percorrer $2^{81}$ subconjuntos: é um único
problema de otimização convexa.

A pergunta óbvia é *por quê*. Por que $|\beta|$ zera e $\beta^2$ não?

---
## 6. Por que o Lasso zera e o Ridge não

### Argumento 1: o caso ortogonal

Quando as colunas de $X$ são ortonormais, os dois problemas têm solução fechada em
função de $\hat\beta^{MQO}$ — e as fórmulas são reveladoras:

$$\hat\beta^{Ridge}_j = \frac{\hat\beta^{MQO}_j}{1+\lambda},
\qquad
\hat\beta^{Lasso}_j = \operatorname{sinal}\big(\hat\beta^{MQO}_j\big)\,
\max\big(|\hat\beta^{MQO}_j| - \tfrac{\lambda}{2},\ 0\big).$$

A do Ridge é uma **multiplicação**: encolhe proporcionalmente, e o produto de um
número não-nulo por $1/(1+\lambda)$ nunca é zero. A do Lasso é uma **subtração com
piso** — o *soft-thresholding*: tudo o que era menor que $\lambda/2$ em módulo
vira exatamente zero.

(O $\lambda/2$ não é errata de digitação: sai de derivar
$\|y - X\beta\|^2 + \lambda\sum_j|\beta_j|$, sem $\tfrac12$ no RSS, que é a
convenção das notas. Com $\tfrac12\|y - X\beta\|^2$ o limiar seria $\lambda$ — e
aí a fórmula do Ridge ao lado passaria a ser $\hat\beta/(1+2\lambda)$.)

In [ ]:
b = np.linspace(-3, 3, 400)
lam = 1.0

fig, ax = subplots(figsize=(6.5, 6))
ax.plot(b, b, "k--", lw=1, label="MQO (sem penalização)")
ax.plot(b, b / (1 + lam), lw=2.5, label=r"Ridge: $\beta/(1+\lambda)$")
ax.plot(b, np.sign(b) * np.maximum(np.abs(b) - lam / 2, 0), lw=2.5,
        label=r"Lasso: $\mathrm{sinal}(\hat\beta)(|\hat\beta|-\lambda/2)_+$")
ax.axhline(0, c="gray", lw=.7)
ax.axvline(0, c="gray", lw=.7)
ax.set_xlabel(r"$\hat\beta^{MQO}$")
ax.set_ylabel("estimativa penalizada")
ax.set_title(r"Caso ortogonal, $\lambda = 1$")
ax.set_aspect("equal")
ax.legend();

O patamar achatado da curva laranja entre $-0{,}5$ e $0{,}5$ — isto é, entre
$-\lambda/2$ e $\lambda/2$ — é a esparsidade. A curva azul
do Ridge passa pela origem, mas só a toca **num ponto**.

### Argumento 2: a geometria

A formulação equivalente das notas troca a penalização por uma **restrição**:

$$\min_\beta \ \|y - X\beta\|^2 \quad \text{sujeito a} \quad
\underbrace{\textstyle\sum_j \beta_j^2 \le t}_{\text{Ridge: bola } \ell_2}
\qquad\text{ou}\qquad
\underbrace{\textstyle\sum_j |\beta_j| \le t}_{\text{Lasso: bola } \ell_1}$$

As curvas de nível do RSS são elipses centradas em $\hat\beta^{MQO}$. A solução
está onde a **menor** elipse toca a região de restrição. E aí está tudo: a bola
$\ell_1$ tem **quinas sobre os eixos**, e uma elipse que se expande tem alta chance
de tocar primeiro numa quina — e uma quina é um ponto onde alguma coordenada é
**exatamente zero**. A bola $\ell_2$ é lisa: o toque acontece num ponto genérico,
com todas as coordenadas não-nulas.

Vamos construir a figura com um exemplo de duas covariáveis. Importante: os pontos
marcados são as soluções **de verdade**, calculadas pelo `scikit-learn` — a
tangência não foi desenhada à mão.

In [ ]:
rng = np.random.default_rng(3)
n2 = 60

z1 = rng.normal(size=n2)
z2 = 0.6 * z1 + 0.8 * rng.normal(size=n2)      # correlacionada com z1
X2 = np.column_stack([z1, z2])
X2 = (X2 - X2.mean(0)) / X2.std(0)
y2 = X2 @ np.array([1.4, 0.45]) + rng.normal(0, 0.8, n2)
y2 = y2 - y2.mean()

beta_ols2 = np.linalg.lstsq(X2, y2, rcond=None)[0]
beta_lasso2 = skl.Lasso(alpha=0.6, fit_intercept=False).fit(X2, y2).coef_
beta_ridge2 = skl.Ridge(alpha=55.0, fit_intercept=False).fit(X2, y2).coef_

print("beta MQO  :", beta_ols2.round(4))
print("beta Lasso:", beta_lasso2.round(4), "  <- segunda coordenada EXATAMENTE zero")
print("beta Ridge:", beta_ridge2.round(4), "  <- nenhuma coordenada zero")

In [ ]:
def rss2(b1, b2):
    return np.sum((y2[:, None, None] - X2[:, 0, None, None] * b1
                   - X2[:, 1, None, None] * b2) ** 2, axis=0)

g1, g2 = np.linspace(-0.4, 2.0, 300), np.linspace(-0.9, 1.5, 300)
G1, G2 = np.meshgrid(g1, g2)
Z = rss2(G1, G2)
theta = np.linspace(0, 2 * np.pi, 400)

fig, axes = subplots(1, 2, figsize=(13, 6))

for ax, beta_hat, tipo in [(axes[0], beta_lasso2, "Lasso"),
                           (axes[1], beta_ridge2, "Ridge")]:
    base = float(rss2(*beta_hat))
    ax.contour(G1, G2, Z, levels=[base * f for f in (1, 1.06, 1.15, 1.28, 1.45)],
               colors="steelblue", linewidths=1.1)

    if tipo == "Lasso":
        t = np.abs(beta_hat).sum()                      # raio l1
        regiao = np.array([[t, 0], [0, t], [-t, 0], [0, -t], [t, 0]])
    else:
        t = np.linalg.norm(beta_hat)                    # raio l2
        regiao = np.column_stack([t * np.cos(theta), t * np.sin(theta)])

    ax.plot(regiao[:, 0], regiao[:, 1], "r-", lw=2)
    ax.fill(regiao[:, 0], regiao[:, 1], "red", alpha=0.12)

    ax.plot(*beta_ols2, "ko", ms=9)
    ax.annotate(r"$\hat\beta^{MQO}$", beta_ols2, textcoords="offset points",
                xytext=(10, 6), fontsize=13)
    ax.plot(*beta_hat, "r*", ms=18)
    ax.axhline(0, c="gray", lw=.7)
    ax.axvline(0, c="gray", lw=.7)
    ax.set_xlabel(r"$\beta_1$")
    ax.set_ylabel(r"$\beta_2$")
    ax.set_title(f"{tipo}: solução em {beta_hat.round(3)}")
    ax.set_aspect("equal")

À esquerda, a elipse toca o losango exatamente na **quina da direita**, onde
$\beta_2=0$. À direita, ela toca o círculo num ponto qualquer da borda, com as duas
coordenadas não-nulas.

Em duas dimensões o argumento pode parecer uma coincidência geométrica. Em 81
dimensões ele fica muito mais forte: a bola $\ell_1$ tem quinas, arestas e faces de
todas as dimensões intermediárias — e a esmagadora maioria da sua fronteira está
sobre algum subespaço coordenado. Tocar "num ponto genérico" da bola $\ell_1$ é,
justamente, tocar num ponto com muitas coordenadas nulas.

A figura acima fixou um $\lambda$. Vale ver a solução se mover quando ele muda: o
mesmo losango e as mesmas elipses, agora para três valores de `alpha`.

In [ ]:
alphas_fig = (0.3, 0.6, 1.2)

fig, axes = subplots(1, 3, figsize=(15, 5.2))
for ax, a in zip(axes, alphas_fig):
    bl = skl.Lasso(alpha=a, fit_intercept=False).fit(X2, y2).coef_
    base = float(rss2(*bl))
    ax.contour(G1, G2, Z, levels=[base * f for f in (1, 1.06, 1.15, 1.28, 1.45)],
               colors="steelblue", linewidths=1.1)
    t = np.abs(bl).sum()
    losango = np.array([[t, 0], [0, t], [-t, 0], [0, -t], [t, 0]])
    ax.plot(losango[:, 0], losango[:, 1], "r-", lw=2)
    ax.fill(losango[:, 0], losango[:, 1], "red", alpha=0.12)
    ax.plot(*beta_ols2, "ko", ms=8)
    ax.plot(*bl, "r*", ms=17)
    ax.axhline(0, c="gray", lw=.7)
    ax.axvline(0, c="gray", lw=.7)
    ax.set_title(f"alpha = {a}   ->   {(bl != 0).sum()} nao-nulo(s)")
    ax.set_xlabel(r"$\beta_1$")
axes[0].set_ylabel(r"$\beta_2$")
fig.tight_layout()

print(f"{'alpha':>7} {'beta_1':>9} {'beta_2':>9} {'nao-nulos':>11}  {'raio l1 (t)':>12}")
print("-" * 54)
for a in alphas_fig:
    bl = skl.Lasso(alpha=a, fit_intercept=False).fit(X2, y2).coef_
    print(f"{a:7.1f} {bl[0]:9.4f} {bl[1]:9.4f} {(bl != 0).sum():11d}  {np.abs(bl).sum():12.4f}")

**É o $\lambda$ pequeno que tira a solução da quina.** Com `alpha=0.3` o losango é
grande o bastante para a elipse encostar nele numa **aresta**, e as duas coordenadas
sobrevivem: $\hat\beta = (1{,}159\,;\,0{,}188)$. Com `0.6` e `1.2` o toque volta
para a quina da direita, e $\hat\beta_2$ é exatamente zero.

A relação é monótona, e agora dá para enunciá-la: **quanto maior o $\lambda$, menor
o losango** — o raio $\ell_1$ cai de $1{,}35$ para $0{,}95$ e depois $0{,}35$ — **e
mais provável que a elipse o encontre primeiro numa quina**, isto é, mais
coeficientes zerados. No limite $\lambda\to\infty$ o losango encolhe até a origem e
todos zeram.

Repare também no coeficiente que sobrevive: $1{,}159 \to 0{,}955 \to 0{,}355$. Zerar
$\hat\beta_2$ não protege $\hat\beta_1$; a penalização segue encolhendo o que
restou. Seleção e encolhimento acontecem ao mesmo tempo, que é exatamente o que a
fórmula do *soft-thresholding* diz.

---
## 7. O Lasso descobre quais coeficientes são nulos?

A figura anterior mostra que o Lasso *pode* zerar coeficientes. Falta a pergunta
que importa: ele zera **os certos**?

Com dados reais isso é inverificável — não sabemos quais covariáveis são
irrelevantes. Então voltamos ao truque da Aula 01: simular, para conhecer a
resposta. Retomamos o $\beta$ esparso da Seção 5 daquele notebook, agora com 20
covariáveis, das quais só 5 têm efeito.

In [ ]:
rng = np.random.default_rng(7)

p = 20
n_grande = 1000

beta_verdadeiro = np.zeros(p)
beta_verdadeiro[[0, 3, 7, 11, 15]] = [3.0, -2.0, 1.5, 2.5, -1.8]

Xsim = rng.normal(size=(n_grande, p))
ysim = Xsim @ beta_verdadeiro + rng.normal(0, 1.0, n_grande)

print("covariaveis com efeito real:", np.flatnonzero(beta_verdadeiro).tolist())
print("as outras 15 sao ruido puro.")

Com $n=1000$ e $p=20$ estamos num regime confortável: o MQO não vai quebrar. A
questão aqui **não é predição** — é seleção.

In [ ]:
mqo_sim   = skl.LinearRegression().fit(Xsim, ysim)
lasso_sim = skl.LassoCV(cv=5, random_state=0).fit(Xsim, ysim)
ridge_sim = skl.RidgeCV(alphas=np.logspace(-3, 3, 50)).fit(Xsim, ysim)

resultado = pd.DataFrame({
    "verdadeiro": beta_verdadeiro,
    "MQO":        mqo_sim.coef_,
    "Ridge":      ridge_sim.coef_,
    "Lasso":      lasso_sim.coef_,
}, index=[f"x{j}" for j in range(p)])

print("coeficientes exatamente nulos:")
print(f"  verdadeiro : {(beta_verdadeiro == 0).sum()} de {p}")
print(f"  MQO        : {(mqo_sim.coef_ == 0).sum()} de {p}")
print(f"  Ridge      : {(ridge_sim.coef_ == 0).sum()} de {p}")
print(f"  Lasso      : {(lasso_sim.coef_ == 0).sum()} de {p}")
resultado.round(3)

In [ ]:
fig, ax = subplots(figsize=(11, 5))
pos = np.arange(p)

ax.stem(pos - 0.22, beta_verdadeiro, linefmt="k-", markerfmt="ko", basefmt=" ",
        label="verdadeiro")
ax.stem(pos, mqo_sim.coef_, linefmt="C0-", markerfmt="C0s", basefmt=" ",
        label="MQO")
ax.stem(pos + 0.22, lasso_sim.coef_, linefmt="C1-", markerfmt="C1^", basefmt=" ",
        label="Lasso (CV)")
ax.axhline(0, c="gray", lw=.8)
ax.set_xticks(pos)
ax.set_xticklabels([f"x{j}" for j in range(p)], fontsize=8)
ax.set_ylabel(r"$\hat\beta_j$")
ax.set_title("Estimativas contra o valor verdadeiro")
ax.legend();

Duas leituras, e a segunda é uma surpresa.

**O MQO** (quadrados azuis) dá a cada covariável de ruído um valor pequeno mas
**não-nulo**. Ele não tem como fazer diferente: nada no critério de mínimos
quadrados premia um zero exato.

**O Lasso** zerou só 2 das 15. Isso contradiz tudo o que dissemos até aqui? Não —
mas obriga a olhar as **magnitudes**, e não a contagem:

In [ ]:
suporte = set(np.flatnonzero(beta_verdadeiro).tolist())
ruido = sorted(set(range(p)) - suporte)

print("coeficientes VERDADEIROS (5):")
print("  Lasso estimou:", np.abs(lasso_sim.coef_[sorted(suporte)]).round(3))
print("  valor real   :", np.abs(beta_verdadeiro[sorted(suporte)]).round(3))
print()
print("coeficientes de RUIDO (15):")
print("  Lasso estimou:", np.abs(lasso_sim.coef_[ruido]).round(3))
print()
print(f"maior falso positivo : {np.abs(lasso_sim.coef_[ruido]).max():.4f}")
print(f"menor coef. real     : {np.abs(lasso_sim.coef_[sorted(suporte)]).min():.4f}")

Os cinco coeficientes reais foram recuperados com precisão. Os treze falsos
positivos são todos **menores que 0,07** — vinte vezes menores que o menor
coeficiente verdadeiro. O Lasso não se confundiu; ele apenas não terminou o
serviço.

Por que não? Porque escolhemos $\lambda$ com `LassoCV`, isto é, **por predição**.
E para predizer bem é barato manter um coeficiente minúsculo: ele quase não
atrapalha. A validação cruzada, otimizando erro quadrático, não tem motivo para
zerá-lo.

Se o objetivo for **selecionar**, o $\lambda$ certo é outro:

In [ ]:
print(f"{'lambda':>8} {'nao-nulos':>11} {'suporte exato?':>16}   selecionadas")
print("-" * 62)
for lam in [0.015, 0.05, 0.10, 0.30, 0.70, 1.00]:
    b = skl.Lasso(alpha=lam).fit(Xsim, ysim).coef_
    sel = set(np.flatnonzero(b).tolist())
    marca = "SIM" if sel == suporte else "nao"
    print(f"{lam:8.3f} {len(sel):11d} {marca:>16}   "
          f"{sorted(sel) if len(sel) <= 8 else '(muitas)'}")

print()
print("lambda escolhido pela CV (predicao):", round(lasso_sim.alpha_, 5))

Qualquer $\lambda$ entre $0{,}10$ e $1{,}00$ — uma faixa larga — recupera
**exatamente** as cinco covariáveis certas, sem nenhum falso positivo. A CV
escolheu $0{,}015$, bem abaixo dessa faixa.

> **A lição.** O $\lambda$ que melhor prediz **não** é o $\lambda$ que melhor
> seleciona. São objetivos diferentes e pedem ajustes diferentes. Se você vai
> reportar "estas são as variáveis importantes", não pode simplesmente aceitar o
> $\lambda$ que a validação cruzada devolveu — ela respondeu a outra pergunta.

Note ainda que as estimativas do Lasso para as covariáveis relevantes são
**ligeiramente menores** em módulo que as verdadeiras (2,94 contra 3,0; 1,74 contra
1,8). Isso é o encolhimento, e é o preço: o Lasso é um estimador **enviesado**.
Trocamos viés por variância, exatamente como a Aula 01 previa.

### E quando os dados são escassos?

Acima, o MQO previa bem; só era ilegível. Agora o regime difícil: $n=40$ para
$p=20$ — duas observações por parâmetro.

In [ ]:
n_pequeno = 40
Xh = rng.normal(size=(n_pequeno, p))
yh = Xh @ beta_verdadeiro + rng.normal(0, 1.0, n_pequeno)

Xh_te = rng.normal(size=(5000, p))
yh_te = Xh_te @ beta_verdadeiro + rng.normal(0, 1.0, 5000)

modelos = {
    "MQO":   skl.LinearRegression(),
    "Ridge": skl.RidgeCV(alphas=np.logspace(-3, 3, 50)),
    "Lasso": skl.LassoCV(cv=5, random_state=0),
}

linhas = []
for nome, m in modelos.items():
    m.fit(Xh, yh)
    linhas.append({
        "modelo": nome,
        "EQM treino": mean_squared_error(yh, m.predict(Xh)),
        "EQM teste": mean_squared_error(yh_te, m.predict(Xh_te)),
        "nao-nulos": int((m.coef_ != 0).sum()),
    })

pd.DataFrame(linhas).set_index("modelo").round(3)

Agora a diferença aparece na predição, e não só na legibilidade: o MQO tem o menor
erro de **treino** e o maior de **teste** — a assinatura do superajuste. Ridge e
Lasso, ao aceitarem um ajuste pior nos dados que já têm, generalizam melhor.

Este é o argumento central da aula em uma tabela.

A tabela acima é um retrato de $n=40$. A pergunta natural é como ela se move com
$n$, e vale medir os dois lados: menos dados ($n=25$, com $p=20$ — quase um
parâmetro por observação) e muitos ($n=500$).

Uma observação de método. Cada linha abaixo é a **média de 20 repetições**, com
amostra nova a cada uma, e vem acompanhada do erro-padrão dessa média. Com $n=25$ um
único sorteio varia demais para se concluir qualquer coisa dele — e sem a barra de
erro não daria para saber quais diferenças da tabela são reais.

In [ ]:
def risco_por_n(n_treino, repeticoes=20):
    """EQM de teste de MQO, Ridge e Lasso com n_treino observacoes.

    Devolve media e erro-padrao da media sobre as repeticoes: com n=25 um
    unico sorteio varia demais para se concluir qualquer coisa dele.
    """
    acumulado = {"MQO": [], "Ridge": [], "Lasso": []}
    for r in range(repeticoes):
        rng_n = np.random.default_rng(100 + r)
        Xtr = rng_n.normal(size=(n_treino, p))
        ytr = Xtr @ beta_verdadeiro + rng_n.normal(0, 1.0, n_treino)
        Xte = rng_n.normal(size=(4000, p))
        yte = Xte @ beta_verdadeiro + rng_n.normal(0, 1.0, 4000)
        for nome, m in [("MQO", skl.LinearRegression()),
                        ("Ridge", skl.RidgeCV(alphas=np.logspace(-3, 3, 50))),
                        ("Lasso", skl.LassoCV(cv=5, random_state=0))]:
            m.fit(Xtr, ytr)
            acumulado[nome].append(mean_squared_error(yte, m.predict(Xte)))
    return {k: (float(np.mean(v)), float(np.std(v, ddof=1) / np.sqrt(len(v))))
            for k, v in acumulado.items()}

print(f"EQM de teste, media +- erro-padrao da media (20 repeticoes), p = {p}\n")
print(f"{'n':>5}   {'MQO':>16} {'Ridge':>16} {'Lasso':>16}   {'MQO/melhor':>11}")
print("-" * 76)
for n_treino in (25, 40, 500):
    r = risco_por_n(n_treino)
    melhor = min(r["Ridge"][0], r["Lasso"][0])
    celulas = "".join(f"{r[k][0]:10.3f} +-{r[k][1]:5.3f}" for k in ("MQO", "Ridge", "Lasso"))
    print(f"{n_treino:>5}   {celulas}   {r['MQO'][0] / melhor:11.2f}")

**A vantagem da regularização cresce quando os dados escasseiam, e evapora quando
sobram.** O MQO custa $3{,}01$ vezes o melhor modelo penalizado com $n=25$;
$1{,}36$ vezes com $n=40$; e $1{,}01$ vez com $n=500$.

O motivo é o da Aula 01. O erro do MQO carrega uma parcela de variância que cresce
com $p/n$: com 20 covariáveis e 25 observações ela domina tudo, e trocar variância
por viés é um negócio excelente. Com $n=500$ a variância do MQO já é pequena, não há
o que comprar, e as três linhas encostam no erro irredutível $\sigma^2 = 1$.

É aqui que as barras de erro fazem falta. Em $n=500$ o Ridge aparece $0{,}001$ acima
do MQO — e o erro-padrão de cada um é $0{,}006$. **Não há diferença nenhuma**: nesse
regime os três empatam, e quem lesse só as médias concluiria besteira. Já em $n=25$,
$2{,}670 \pm 0{,}332$ contra $8{,}042 \pm 1{,}357$ é uma separação que nenhuma barra
de erro desfaz.

A lição prática: regularizar **não** é sempre melhor. É melhor quando existe
variância para comprar, e com $n$ grande em relação a $p$ não existe.

Um último aviso, para não generalizar demais: o Lasso vence o Ridge nos três
regimes, mas isso não é uma verdade geral. O $\beta$ verdadeiro desta simulação é
**esparso por construção** — 15 dos 20 coeficientes são exatamente zero —, que é
justamente a situação feita para a $\ell_1$ brilhar. Com um $\beta$ denso a ordem
costuma se inverter.

---
## 8. Comparação final

Falta juntar tudo em dados de verdade. Para a comparação valer alguma coisa, os
três modelos precisam ser julgados em observações que nenhum deles viu: separamos
30% dos supercondutores e só voltamos a tocá-los na última linha desta seção.

In [ ]:
X_tr, X_te, y_tr, y_te = skm.train_test_split(X, y, test_size=0.3, random_state=0)

print("treino:", X_tr.shape, " teste:", X_te.shape)

Cada modelo penalizado precisa de um $\lambda$, e escolhê-lo olhando o teste seria
trapaça. Usamos `RidgeCV` e `LassoCV`, como na Seção 7: eles varrem uma grade
internamente e ficam com o valor de melhor desempenho, olhando **só** o treino.
Como essa varredura é feita — validação cruzada — é assunto da Aula 03; aqui as
duas entram como caixa-preta.

A padronização vai **dentro** do `Pipeline`, pelo motivo da Seção 4: assim ela é
reajustada em cada ajuste, em vez de ver o banco inteiro de uma vez.

In [ ]:
mqo_final = skl.LinearRegression().fit(X_tr, y_tr)

ridge_cv = Pipeline([("escala", StandardScaler()),
                     ("ridge", skl.RidgeCV(alphas=np.logspace(-3, 3, 50)))]).fit(X_tr, y_tr)

lasso_cv = Pipeline([("escala", StandardScaler()),
                     ("lasso", skl.LassoCV(cv=5, random_state=0))]).fit(X_tr, y_tr)

ridge_final = ridge_cv.named_steps["ridge"]
lasso_final = lasso_cv.named_steps["lasso"]

print("lambda escolhido (Ridge):", round(ridge_final.alpha_, 4))
print("lambda escolhido (Lasso):", round(lasso_final.alpha_, 5))

In [ ]:
comparacao = []
for nome, modelo, coefs in [
    ("MQO",         mqo_final, mqo_final.coef_),
    ("Ridge (CV)",  ridge_cv,  ridge_final.coef_),
    ("Lasso (CV)",  lasso_cv,  lasso_final.coef_),
]:
    comparacao.append({
        "modelo": nome,
        "EQM treino": mean_squared_error(y_tr, modelo.predict(X_tr)),
        "EQM teste": mean_squared_error(y_te, modelo.predict(X_te)),
        "nao-nulos": int((coefs != 0).sum()),
    })

pd.DataFrame(comparacao).set_index("modelo").round(3)

Uma leitura honesta desta tabela precisa reconhecer o seguinte: aqui
$n = 14\,884$ e $p = 81$. Estamos **longe** do regime de dimensão alta, e nesse
regime o MQO é difícil de bater em erro de predição — a regularização tem pouco a
corrigir. O Ridge fica meio ponto de EQM à frente dele, $307{,}3$ contra $307{,}8$,
o que é 0,15% e não é ganho de nada. O Lasso **perde**: $318{,}8$, 3,6% pior.

O ganho do Lasso está na última coluna — 66 covariáveis em vez de 81 —, e é por ela
que se paga esses 3,6%.

Isso não enfraquece a aula, delimita-a. A regularização vale a pena quando:

- $p$ é grande em relação a $n$ (Seção 7, regime $n=40$);
- há colinearidade forte (Seção 3);
- ou quando **interpretabilidade** é parte do produto, e 15 covariáveis a menos
  valem uma perda pequena de erro.

Na Aula 05 veremos o outro lado: com covariáveis genuinamente irrelevantes, a
regularização deixa de ser conveniência e passa a ser necessidade.

---
## Resumo

| Conceito | Onde apareceu | O que vimos |
|---|---|---|
| $\hat\beta^{MQO}=(X^\top X)^{-1}X^\top Y$ | §2 | idêntico ao `scikit-learn`; resolva o sistema, não inverta |
| colinearidade | §3 | $\operatorname{cond}(X^\top X)\approx10^{13}$; VIF na casa dos milhares |
| $p > n$ | §3 | $X^\top X$ singular; infinitas soluções com RSS $=0$ e péssima generalização |
| Ridge ($\ell_2$) | §4 | encolhe todos os coeficientes; **nenhum** vira zero |
| Lasso ($\ell_1$) | §5 | encolhe **e seleciona**: de 81 a 0 covariáveis conforme $\lambda$ |
| por que zera | §6 | *soft-thresholding* no caso ortogonal; quinas da bola $\ell_1$ |
| recuperação do suporte | §7 | Lasso acha as 5 covariáveis reais entre 20; MQO não zera nada |
| comparação final | §8 | com $n\gg p$ o MQO não é batido em EQM; o ganho do Lasso está nas covariáveis que ele dispensa |

**Leitura recomendada.** [AME] Capítulos 2 e 3 — §3.3 (Lasso) e §3.4 (Ridge); a
§3.6 (interpretação bayesiana) está nas notas, mas não neste notebook. [ISLP]
Capítulo 3 (regressão linear) e Capítulo 6, §6.1 (seleção de subconjuntos) e
§6.2 (*shrinkage*); o laboratório §6.5.2 é a fonte direta das Seções 4 e 5
deste notebook.

**Para praticar.** `Lista teorica 02.pdf` (teórica, com gabarito) e
`Lista prática 02.ipynb` (prática, para completar as lacunas), nesta mesma
pasta.

**A seguir.** A Aula 03 formaliza o que aqui ficou por conta do `RidgeCV` e do
`LassoCV`: como estimar o risco a partir de uma única amostra, quantas dobras
usar, e por que a validação cruzada funciona.